In [8]:

from pyspark.sql.session import SparkSession
import pyspark.sql.functions as f

import os
from dotenv import load_dotenv

env = load_dotenv(dotenv_path='/mnt/d/linux/project/vars/.env')

#env vars for postgresql connection
mssqlsource = os.getenv('MSSQLSOURCE')
mssqluser = os.getenv('MSSQL_SPARK_USER')
mssqlpwd = os.getenv('MSSQL_SPARK_PWD')
mssqldriver = os.getenv('MSSQL_DRIVER')
sqlserverjar = os.getenv('SQLSERVER_JAR')
mssql_dwdest = os.getenv('MSSQL_CONN')

In [9]:
#create spark session

spark = ((
        SparkSession
            .builder
            .appName('stock_holding')
            .config('spark.jars', f'{sqlserverjar}')
            .config('spark.executor.extraClassPath', f'{sqlserverjar}')
            .config('spark.driver.extraClassPath', f'{sqlserverjar}')
        .getOrCreate()
))

In [10]:
tmpstockholding_frame = (( spark
                                .read
                                .format('jdbc')
                                .option('url', mssqlsource)
                                .option('driver', mssqldriver)
                                .option('user', mssqluser)
                                .option('password', mssqlpwd)
                                .option('query',
                                        
                                        """ 
                                            select

                                                stockitemid,
                                                binlocation,
                                                reorderlevel,
                                                lasteditedby,
                                                lastcostprice,
                                                quantityonhand,
                                                laststocktakequantity,
                                                targetstocklevel
                                            
                                            from warehouse.stockitemholdings
                                        
                                        """
                                        )
                            
)).load()

In [11]:
#rename columns

itemholding  = (( 
                tmpstockholding_frame
                    .withColumnRenamed('stockitemid', 'stockholdingid')
                    .withColumnRenamed('binlocation', 'shelflocation')
                    .withColumnRenamed('quantityonhand', 'availabeqtd')
                    .withColumnRenamed('laststocktakequantity', 'laststockqtd')
    ))

itemholding.show()

+--------------+-------------+------------+------------+-------------+-----------+------------+----------------+
|stockholdingid|shelflocation|reorderlevel|lasteditedby|lastcostprice|availabeqtd|laststockqtd|targetstocklevel|
+--------------+-------------+------------+------------+-------------+-----------+------------+----------------+
|             1|          L-1|          20|          16|         9.50|     175609|      171341|             100|
|             2|          L-1|          20|           3|         9.50|     165538|      161435|             100|
|             3|          L-2|          10|           3|        11.25|     253190|      246900|             120|
|             4|          L-3|           5|           3|        12.00|     208109|      202964|             100|
|             5|          L-3|           5|           3|        12.00|     199064|      194162|             100|
|             6|          L-3|           5|           3|        12.00|     196995|      192127| 

In [13]:
((
    itemholding
        .write
        .format('jdbc')
        .option('url', mssql_dwdest)
        .option('driver', mssqldriver)
        .option('user', mssqluser)
        .option('password', mssqlpwd)
        .option('dbtable', 'dimension.warehouse')
    .save(mode='overwrite')
))

In [ ]:
spark.stop()